<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 70 Classic

The notebook checks the classic look, the Windows 98 look of 3D-Baukasten 1.3 that the page switches
to: the windows float at the places `Form_Set` of `3D_bau.frm` computes for the screen, the four
views are squares of half the shorter side of Form1 as `ViewControl` makes them, the fields carry
the colours of 1.3, and the renderer draws black views with the white raster and wire bodies. The
page opens at 1024 by 768, the resolution the thesis recommends.

In [ ]:
width = 1024
height = 768
look = 'classic'

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, theme=look)
page.open()
check(page.errors() == [], 'the page starts without an error', page.errors())

## 1. The places of Form_Set

At a screen of W by H pixels `Form_Set` puts F3 at 0, 0 with 48 by 500, F2 under it down to H − 27,
Form1 at 48, 0 with W − 300 by H − 50, F4 at W − 252, 0 with 252 by 580, F6 under F4 at a top of 580,
and F5 at the bottom of Form1 in its width.

In [ ]:
W, H = width, height
expected = {
    '#f3': (0, 0, 48, 500),
    '#f2': (0, 500, 48, H - 527),
    '#form1': (48, 0, W - 300, H - 50),
    '#f4': (W - 252, 0, 252, 580),
    '#f6': (W - 252, 580, 252, 200),
}
for selector, box in expected.items():
    got = page.rect(selector)
    check(all(abs(a - b) <= 2 for a, b in zip(got, box)), f'{selector} sits where Form_Set puts it', got, box)
f5 = page.rect('#f5')
check(abs(f5[0] - 48) <= 2 and abs(f5[2] - (W - 300)) <= 2 and abs(f5[3] - 35) <= 2, 'F5 is 35 pixels high in the width of Form1', f5)

## 2. The views of ViewControl

Each view is a square of F / 2, where F is the shorter side of Form1, the Kreuzriss at the top left,
the Aufriss beside it, the Schrägriss under the Kreuzriss and the Grundriss under the Aufriss.

In [ ]:
v = {n: page.rect(f'.view[data-view="{n}"]') for n in range(4)}
side = v[1][2]
check(all(abs(b[2] - side) <= 1 and abs(b[3] - side) <= 1 for b in v.values()), 'four squares of one size', v)
check(abs(v[0][0] - (v[1][0] + side)) <= 1 and abs(v[3][1] - (v[1][1] + side)) <= 1 and abs(v[2][0] - v[0][0]) <= 1 and abs(v[2][1] - v[3][1]) <= 1, 'the quadrants of ViewControl', v)
views = page.rect('#views')
check(abs(side - min(views[2], views[3]) // 2) <= 1, 'the side is half the shorter side of the area of the views', side, views)

## 3. The colours of 1.3

The fields x, y and z of F5 have red, yellow and green digits on black, the zoom and the status white
on the blue `&H00C00000&`, the fields of the position and the size yellow on black, the turns red,
and the frame GEOMETRIEDATEN is teal.

In [ ]:
color = lambda s, p='color': page.eval("([s, p]) => getComputedStyle(document.querySelector(s))[p]", [s, p])
check(color('#tx') == 'rgb(255, 0, 0)' and color('#ty') == 'rgb(255, 255, 0)' and color('#tz') == 'rgb(0, 255, 0)', 'red, yellow and green digits', color('#tx'), color('#ty'), color('#tz'))
check(color('#tx', 'backgroundColor') == 'rgb(0, 0, 0)', 'on black', color('#tx', 'backgroundColor'))
check(color('#tstatus', 'backgroundColor') == 'rgb(0, 0, 192)' and color('#tzoom', 'backgroundColor') == 'rgb(0, 0, 192)', 'zoom and status on the blue of 1.3', color('#tstatus', 'backgroundColor'))
check(color('#t1-1') == 'rgb(255, 255, 0)' and color('#t1-7') == 'rgb(255, 0, 0)', 'yellow position, red turn', color('#t1-1'), color('#t1-7'))
check(color('#frame1', 'backgroundColor') in ('rgb(0, 128, 128)', 'rgb(0, 192, 192)'), 'the teal frame GEOMETRIEDATEN', color('#frame1', 'backgroundColor'))
check(color('#form1 > .win-title', 'backgroundColor') == 'rgb(128, 0, 128)', 'the purple caption of the thesis', color('#form1 > .win-title', 'backgroundColor'))
zoom = page.eval("() => [document.querySelector('#tzoom').value, bausteine.app.zoomf]")
check(zoom[0] == f'{float(f"{zoom[1] / 10:.7g}")}'.replace('.', ','), 'the 2D zoom shows the raw zoomf / 10 of 1.3 with a comma', zoom)
page.page.click('#d2-zoomout'); page.settle(100)
zoom = page.eval("() => [document.querySelector('#tzoom').value, bausteine.app.zoomf]")
check(zoom[0] == f'{float(f"{zoom[1] / 10:.7g}")}'.replace('.', ',') and ',' in zoom[0], 'and after a step of the ladder', zoom)

## 4. The tool buttons and their icons

The buttons of F2 and F3 are 39 by 38 pixels with a picture of 32 by 32 in the sixteen colours of the
Windows palette.

In [ ]:
sizes = page.eval("() => [...document.querySelectorAll('#tb1 .tb-button, #tb2 .tb-button')].map((b) => { const r = b.getBoundingClientRect(); return [Math.round(r.width), Math.round(r.height)]; })")
check(all(s == [39, 38] for s in sizes), 'every button is 39 by 38', sizes)
palette = {'#000000', '#800000', '#008000', '#808000', '#000080', '#800080', '#008080', '#c0c0c0', '#808080', '#ff0000', '#00ff00', '#ffff00', '#0000ff', '#ff00ff', '#00ffff', '#ffffff'}
fills = page.eval("() => [...document.querySelectorAll('#tb1 svg [fill], #tb2 svg [fill]')].map((e) => e.getAttribute('fill').toLowerCase())")
outside = sorted({f for f in fills if f not in palette and f != 'none'})
check(fills and outside == [], 'the icons use the sixteen colours alone', outside)

## 5. The drawing of 1.3

The renderer draws the views black with a white dot at every unit and the bodies as wire models in
their colours. The picture of the Kreuzriss is read back from the worker.

In [ ]:
path, stats = page.screenshot('70_classic')
shot = page.eval('''async () => { const s = await bausteine.snapshot(); const px = new Uint8Array(s.pixels.buffer);
  const r = document.querySelector('.view[data-view="1"]').getBoundingClientRect(); const box = document.querySelector('#views').getBoundingClientRect();
  const k = s.width / box.width; let black = 0, white = 0, other = 0, n = 0;
  for (let y = Math.round((r.top - box.top + 4) * k); y < (r.bottom - box.top - 4) * k; y += 3)
    for (let x = Math.round((r.left - box.left + 4) * k); x < (r.right - box.left - 4) * k; x += 3) {
      const i = (y * s.width + x) * 4; const [R, G, B] = [px[i], px[i + 1], px[i + 2]]; n++;
      if (R + G + B < 30) black++; else if (R > 225 && G > 225 && B > 225) white++; else other++; }
  return { black: black / n, white: white / n, other: other / n }; }''')
check(shot['black'] > 0.8, 'the Kreuzriss is black', shot)
check(shot['white'] > 0.001, 'with the white raster', shot)
check(shot['other'] > 0.0005, 'and the coloured wires of the bodies', shot)

## 6. The password of the tab Einstellung

The field shows a `#` for every character, `PasswordChar` of `F4.frm`, and 2002 unlocks the paths.

In [ ]:
page.page.click('#sstab1 [data-tab="einstellung"]'); page.settle(200)
page.page.click('#tpasswd')
page.page.keyboard.type('2002')
check(page.eval("() => document.querySelector('#tpasswd').value") == '####', 'the field shows # for each character', page.eval("() => document.querySelector('#tpasswd').value"))
check(page.eval("() => document.querySelector('#tbau-home').disabled") is False, '2002 unlocks the paths')
page.page.click('#bdatei-def'); page.settle(200)
page.page.click('#sstab1 [data-tab="koerper"]')

## 7. Back to the modern look

The switch works both ways, the views follow the look, and no window keeps a place of the classic look.

In [ ]:
page.eval("() => bausteine.setLook('modern')")
page.settle(600)
check(page.eval("() => document.documentElement.dataset.look") == 'modern', 'the modern look')
check(page.eval("() => document.querySelector('#form1').style.left") == '', 'no window keeps an inline place')
page.eval("() => bausteine.setLook('classic')")
page.settle(300)
check(page.errors() == [], 'no error on the way', page.errors())

In [ ]:
page.close()
site.stop()